# Room Layout Optimization - Experiment Analysis

Loads the JSON written by the `scripts/run_e*_*.py` experiments and plots the headline results.
Run the experiment scripts first (see the README) so the result files exist.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPERIMENTS = ROOT / 'results' / 'experiments'


def load(relative: str):
    path = EXPERIMENTS / relative
    if not path.exists():
        print(f'missing {path} - run the corresponding script first')
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def delta(record):
    return record.get('delta', record.get('improvement', 0.0))

## E3 / E4 - optimizer comparison

Mean score improvement and runtime per algorithm.

In [ ]:
def plot_optimizers(payload, title):
    if payload is None:
        return
    algorithms = payload['algorithms']
    records = payload['records']
    deltas = [sum(delta(r) for r in records if r['algorithm'] == a) / max(1, sum(1 for r in records if r['algorithm'] == a)) for a in algorithms]
    runtimes = [sum(r['runtime_s'] for r in records if r['algorithm'] == a) / max(1, sum(1 for r in records if r['algorithm'] == a)) for a in algorithms]
    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(algorithms, deltas, color='#3478f6')
    axes[0].set_title('Mean score improvement')
    axes[0].set_ylabel('delta')
    axes[1].bar(algorithms, runtimes, color='#f59222')
    axes[1].set_title('Mean runtime (s)')
    figure.suptitle(title)
    figure.tight_layout()


plot_optimizers(load('e3_optimization/e3_optimization.json'), 'E3 optimizer comparison')
plot_optimizers(load('e4_optimizers/e4_optimizers.json'), 'E4 optimizer comparison')

## E5 - weight sensitivity

Score of the resulting layout under the baseline scorer for each weight variant.

In [ ]:
e5 = load('e5_weights/e5_weights.json')
if e5 is not None:
    summary = e5['summary']
    variants = list(summary)
    values = [summary[v]['mean_baseline_score'] for v in variants]
    figure, ax = plt.subplots(figsize=(max(10, 0.6 * len(variants)), 4))
    ax.bar(variants, values, color='#2eb85c')
    ax.set_ylabel('mean baseline score')
    ax.set_title('E5 weight sensitivity')
    ax.tick_params(axis='x', rotation=55, labelsize=7)
    figure.tight_layout()

## E6 - robustness

Mean improvement as injected floor-plan noise grows.

In [ ]:
e6 = load('e6_robustness/e6_robustness.json')
if e6 is not None:
    summary = e6['summary']
    levels = [float(level) for level in summary]
    figure, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(levels, [summary[f'{level:.2f}']['mean_delta'] for level in levels], marker='o', label='mean delta')
    ax.plot(levels, [summary[f'{level:.2f}']['improved_rate'] for level in levels], marker='s', label='improvement rate')
    ax.set_xlabel('position noise sigma (m)')
    ax.set_title('E6 robustness')
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.legend()

## E7 - term ablation

Full-scorer feasibility and violation counts when one scoring term is removed from the objective.

In [ ]:
e7 = load('e7_ablation/e7_ablation.json')
if e7 is not None:
    summary = e7['summary']
    variants = list(summary)
    figure, ax = plt.subplots(figsize=(max(10, 0.6 * len(variants)), 4))
    ax.bar(variants, [summary[v]['mean_full_score'] for v in variants], color='#9b59b6')
    ax.set_ylabel('mean full score')
    ax.set_title('E7 term ablation')
    ax.tick_params(axis='x', rotation=55, labelsize=7)
    figure.tight_layout()